# EDA( Understanding data ): bussiness goal: Predict Low Customer Review 
##### which orders are likely to receive a bad review before the review is submitted
## What does one row in our data represent?
### important : We are predicting bad reviews at the time an order is placed

In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns",200)
pd.set_option("display.width",200)

In [ ]:
orders=pd.read_csv("C:/Users/lahar/olist-ml-enterprise/data/raw/olist_orders_dataset.csv")
orders.head()

In [ ]:
# here we are checking if each order_id is in one row (like not in multiple rows)
print("total rows", len(orders))
# the line below counts how many unique order IDs exist
print("Unique order_id:",orders["order_id"].nunique())

##### ✅ Conclusion:

Each row in orders = one unique order
No duplicates at order level.
This table is clean structurally.

That means:
orders will become the backbone of our ML dataset.

So from orders:

###### order_id → connects to:

order_items

order_payments

order_reviews

###### customer_id → connects to:

customers

🚨 Now Important Step: Fix Date Columns

Right now, all those date columns are probably stored as strings.

If we don’t convert them properly:

We can’t calculate delivery time

We can’t calculate delay

We can’t extract month/day

We can’t build time-based features

So this step is foundational.

In [ ]:
# inspecting date column before converting 
print(orders["order_purchase_timestamp"].head(10))
# for gettinf 10 random rows from anywhere in dataset 
# professional check like verytfying full coloumn
print(orders["order_purchase_timestamp"].sample(10))

In [ ]:
# professional check like verytfying full coloumn
print(orders["order_purchase_timestamp"].dtype)
# checking unique length of strin s
# professional check like verytfying full coloumn
''' if format is different then string length might differ'''
print(orders["order_purchase_timestamp"].str.len().value_counts())
'''🧠 What This Does

".str.len() → counts number of characters in EACH row

.value_counts() → counts how many rows have each length"
'''

In [ ]:
# converting into date 
orders["order_purchase_timestamp"]=pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders["order_purchase_timestamp"].dtype

In [ ]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

orders[date_cols].dtypes

In [ ]:
print("before adding a new column:\n")
print(orders.head())
print("\nafter adding a new column\n")
orders["delivery_time_days"]=(orders["order_delivered_customer_date"]-orders["order_purchase_timestamp"]).dt.days
print(orders.head())

In [ ]:
print(orders[["delivery_time_days"]].isna().sum())
print(orders[["delivery_time_days"]].notna().sum())

In [ ]:
# verifying the status of not delivered items 
orders.loc[orders["delivery_time_days"].isna(),"order_status"].value_counts()

In [ ]:
# verigyuing those 8 rows which are delivered and NAN values
orders.loc[(orders["order_status"]=="delivered")&(orders["delivery_time_days"].isna()),["order_id", "order_status", "order_purchase_timestamp", "order_delivered_customer_date"]]

In [ ]:
# LOADING REVIEWS TABLE
reviews=pd.read_csv("C:/Users/lahar/olist-ml-enterprise/data/raw/olist_order_reviews_dataset.csv")
reviews.head()

In [ ]:
print("Shape (rows, cols):", reviews.shape)
print("\nColumns:\n", reviews.columns)

print("\nDtypes:\n", reviews.dtypes)
print("\n Missing values :\n",reviews.isna().sum().sort_values(ascending=False).head(10))
reviews.head(10)

In [ ]:
# checking if review id is a primary key
print("total rows:",len(reviews))
print("unique review id", reviews["review_id"].nunique())

In [ ]:
reviews[reviews["review_id"].duplicated(keep=False)].sort_values("review_id").head(10)

In [ ]:
result=reviews.groupby("order_id")["review_score"].nunique().value_counts()
print(result)
'''
This tells us:

For each order_id,
how many unique review_scores exist.'''

🎯 What We Can Do

For those 202 problematic orders, we could:

Option A — Keep the latest review (most common)

Use the review with latest timestamp.

Option B — Keep the lowest score (conservative)

Assume worst customer experience.

Option C — Average the scores (not ideal for classification)
Option D — Drop those 202 orders (simplest)

Since 202 is tiny,
dropping them is statistically safe.

In [ ]:
problem_orders=reviews.groupby("order_id")["review_score"].nunique()
problem_orders=problem_orders[problem_orders>1]
problem_orders.head()

In [ ]:
reviews[reviews["order_id"] == "013056cfe49763c6f66bda03396c5ee3"]

In [ ]:
reviews_sorted = reviews.sort_values("review_answer_timestamp")

reviews_latest = reviews_sorted.drop_duplicates(
    subset="order_id",
    keep="last"
)
print("Original review rows:", len(reviews))
print("After keeping latest per order:", len(reviews_latest))

In [ ]:
reviews_latest["order_id"].nunique()

In [ ]:
reviews_latest["low_review"] = (reviews_latest["review_score"] <= 2).astype(int)

reviews_latest["low_review"].value_counts()

In [ ]:
orders_with_reviews = orders.merge(
    reviews_latest[["order_id", "review_score", "low_review"]],
    on="order_id",
    how="inner"
)

print("order_with_reviews rows:", len(orders_with_reviews))
print("unique order_id:", orders_with_reviews["order_id"].nunique())
orders_with_reviews.head()

In [ ]:
print(orders_with_reviews["order_status"].value_counts())
print("\nMissing delivery_time_days:", orders_with_reviews["delivery_time_days"].isna().sum())

In [ ]:
orders_with_reviews = orders_with_reviews[orders_with_reviews["order_status"] == "delivered"].copy()
print("After keeping delivered:", len(orders_with_reviews))
print("Missing delivery_time_days:", orders_with_reviews["delivery_time_days"].isna().sum())

In [ ]:
orders_with_reviews = orders_with_reviews[orders_with_reviews["delivery_time_days"].notna()].copy()

print("Final dataset size:", len(orders_with_reviews))
print("Missing delivery_time_days:", orders_with_reviews["delivery_time_days"].isna().sum())

##### INSPECTING THE ORDER ITEMS TABLE 

In [ ]:
# Does one order appear multiple times in this table?
order_items=pd.read_csv("C:/Users/lahar/olist-ml-enterprise/data/raw/olist_order_items_dataset.csv")
order_items["order_id"].value_counts().head() 

In [ ]:
#If order_id appears 21 times, what exactly is different in those 21 rows?
order_items.groupby("order_id")["product_id"].nunique().sort_values(ascending=False).head(10)

In [ ]:
# just seeing the item id :8272b63d03f5f79c56e9e4120aec44ef how many unique product ids aree existing  
order_items[order_items["order_id"] == "8272b63d03f5f79c56e9e4120aec44ef"]["product_id"].nunique()

In [ ]:
'''
If the same product appears multiple times in an order, how is quantity represented?
to check this let us run:
'''
order_items["order_item_id"].value_counts().head()

In [ ]:
order_items.loc[order_items["order_id"]=="8272b63d03f5f79c56e9e4120aec44ef",["order_id", "order_item_id", "product_id", "seller_id", "price", "freight_value"]].sort_values("order_item_id")

In [ ]:
# Does an order contain items from multiple sellers?
order_items.groupby("order_id")["seller_id"].nunique().sort_values(ascending=False).head(10)

In [ ]:
#Do different products in the same order come from different sellers, or does one seller sell all the products in that order?
order_items.groupby(["order_id", "product_id"])["seller_id"].nunique().sort_values(ascending=False).head(20)
#B) Find orders where multiple sellers exist (this is what you’re chasing)
order_level = order_items.groupby("order_id").agg(
    n_items=("order_item_id", "count"),
    n_unique_products=("product_id", "nunique"),
    n_unique_sellers=("seller_id", "nunique")
)
order_level.sort_values("n_unique_sellers",ascending=False).head(20)

In [ ]:
order_level["n_unique_sellers"].value_counts().sort_index()

In [ ]:
order_items.head(20)

In [ ]:
# BASED ON ABOVE DATA WE HAVE UNDERSTOOD LIKE:
'''
"1 order → many items
1 order → many products
1 order → possibly many sellers"
'''
# creating order level features 
order_item_features=order_items.groupby("order_id").agg(
    n_items=("order_item_id","count"),
    n_unique_products=("product_id","nunique"),
    n_unique_sellers=("seller_id","nunique"),
    total_order_values=("price","sum"),
    total_freight_value=("freight_value","sum")
)
order_item_features.head()

In [ ]:
# merging the features table if order_items: order_item_features on left join because 
# we dont wanna miss any training data right soo
orders_with_items = orders.merge(order_item_features, on="order_id", how="left")
# checking the no of rows
print("rows:", len(orders_with_items))
# checking the no of unique orderids in orderitems so if both no of rows match then we can say no duplication
print("unique order_id:", orders_with_items["order_id"].nunique())
orders_with_items.sample(10)

In [ ]:
#now seeing how many null values are present in the table like for each deature 
orders_with_items[["n_items","n_unique_products","n_unique_sellers",
                   "total_order_values","total_freight_value"]].isna().sum()

In [ ]:
orders_with_items.loc[
    orders_with_items["n_items"].isna(), "order_status"
].value_counts()

In [ ]:
orders_with_items["delivery_time_days"].describe()

In [ ]:
orders_with_items = orders_with_items[
    orders_with_items["order_status"] == "delivered"
]
orders_with_items["order_status"].value_counts()
orders_with_items.shape

In [ ]:
# we are creating delivery delay feature which is an important feature for review affecting 
orders_with_items["delivery_delay_days"]=(orders_with_items["order_delivered_customer_date"]-orders_with_items["order_estimated_delivery_date"]).dt.days
orders_with_items["delivery_delay_days"].describe()

In [ ]:
# removing or ignoring or dropping the days whose timestamps are mising
# like having devilvery time days who values are not null 
orders_with_items = orders_with_items[
    orders_with_items["delivery_time_days"].notna()
].copy()
orders_with_items

In [ ]:
order_payments=pd.read_csv("C:/Users/lahar/olist-ml-enterprise/data/raw/olist_order_payments_dataset.csv")
print("rows:", len(order_payments))
print("unique order_id:", order_payments["order_id"].nunique())
#checking if the order_id is unique because we are seeing if thereis one row per one order existence
order_payments["order_id"].value_counts().head(10)

In [ ]:
# seeing how many payment types we had: 
print(order_payments["payment_type"].value_counts())
print(order_payments["payment_installments"].describe())
order_payments["order_id"].value_counts().head(10)

In [ ]:
print(order_payments["payment_installments"].value_counts().sort_index())
order_payments["payment_installments"].describe()

In [ ]:
# see the documentation for undersatinding why we are creating thsi features 
payment_features = order_payments.groupby("order_id").agg(
    n_payments=("payment_sequential", "max"),
    total_payment_value=("payment_value", "sum"),
    max_installments=("payment_installments", "max")
)
'''
Pandas automatically makes order_id the index, not a column. after above code;
So your dataframe now looks like this internally:

index: order_id
columns:
    n_payments
    total_payment_value
    max_installments

That means:

order_id is NOT a column anymore

So when you run:

payment_features["order_id"]

pandas throws:

KeyError: 'order_id'

because it only exists as an index, not a column.

Two ways to fix it
Option 1 (Best for ML datasets)

Convert the index back to a column.

payment_features = payment_features.reset_index()

Now the dataframe becomes:

order_id	n_payments	total_payment_value	max_installments

Then your check will work:

print(payment_features["order_id"].nunique())
print(len(payment_features))
Option 2 (without resetting index)

You could check directly using the index:

print(payment_features.index.nunique())
print(len(payment_features))

But in ML pipelines we usually reset the index, because we will later merge using order_id.

So do this:

payment_features = payment_features.reset_index()
'''
# reseeting index to column
payment_features = payment_features.reset_index()
# seeing if no of ids is equal to no of rows for checking one row per one order thing 
print(payment_features["order_id"].nunique())
print(len(payment_features))
# seeing if both are equal 
payment_features.head()


#### MAKING A BIG CHANGE HERE

From now on, your main dataset should be:

✅ orders_with_reviews (because it contains the target)


And then you add features into it:

•	add order_item_features

•	add payment_features

So you end with one final dataset:

✅ 1 row per order + target + all features


##### SO FAR WHAT WE HAVE DONE IS :

A) Orders + timestamps + delivery_time_days

You loaded orders, converted timestamp columns to datetime, created:

delivery_time_days = delivered_customer_date - purchase_timestamp

You also checked missingness and order_status for NaNs. ✅

B) Reviews (target)

You loaded reviews, found duplicate reviews per order, kept the latest by review_answer_timestamp, created:

low_review = (review_score <= 2).astype(int)

Then you created:

orders_with_reviews = orders INNER JOIN reviews_latest

Then you filtered:

only order_status == "delivered"

dropped rows where delivery_time_days is null

✅ This is your best base training dataset.

C) Order items features

You built order_item_features by aggregating order_items per order:

n_items, n_unique_products, n_unique_sellers, total_order_values, total_freight_value

Then you merged into orders (not into orders_with_reviews) to create:

orders_with_items = orders LEFT JOIN order_item_features

Then filtered delivered, created delivery_delay_days, dropped delivery_time_days NaNs. ✅

What to do next (to fix the confusion)
The best approach

From now on, your main dataset should be:

✅ orders_with_reviews (because it contains the target)

And then you add features into it:

add order_item_features

add payment_features

So you end with one final dataset:

✅ 1 row per order + target + all features

In [ ]:
# CREATING A MODEL DATAFRAME 
'''
HERE 
i am trying to make orders_with_reviews as main base table and add remaing features to it  USING JOINS
'''
#print(orders_with_reviews.columns)
#print(order_item_features.columns)
print("this is no of rows and unique count before merging with order_item features")
print("rows:", len(orders_with_reviews))
print("unique order_id:", orders_with_reviews["order_id"].nunique())
model_df=orders_with_reviews.merge(
    order_item_features,
    on="order_id",
    how="left"
)
print("this is no of rows and unique count after merging with order_item features")
print("rows:", len(model_df))
print("unique order_id:", model_df["order_id"].nunique())
#print(model_df.columns)
model_df[["n_items","n_unique_products","n_unique_sellers","total_order_values","total_freight_value"]].isna().sum()

In [ ]:
# ADDING PAYMENT FEATURES TO MODEL DF 
model_df = model_df.merge(payment_features, on="order_id", how="left")

print("rows:", len(model_df))
print("unique order_id:", model_df["order_id"].nunique())

model_df[["n_payments","total_payment_value","max_installments"]].isna().sum()

In [ ]:
# IDENTIFYING THE MISSING RECORD WHICH HAD NULL VALUES 
missing_payment_orders = model_df.loc[model_df["n_payments"].isna(), "order_id"]
missing_payment_orders

In [ ]:
# getting the information of row that has null values 
missing_payment_row = model_df.loc[model_df["n_payments"].isna(), :]
missing_payment_row.T

In [ ]:
# since there is one data row that have 3 values that are null let us not drop and fill it with existing dat a
print(model_df["n_payments"].value_counts().head())
print(model_df["max_installments"].value_counts().head())
most_common_payments = model_df["n_payments"].mode()[0]
most_common_installments = model_df["max_installments"].mode()[0]

print(most_common_payments)
print(most_common_installments)
model_df["n_payments"] = model_df["n_payments"].fillna(most_common_payments)
model_df["max_installments"] = model_df["max_installments"].fillna(most_common_installments)

model_df["total_payment_value"] = model_df["total_payment_value"].fillna(
    model_df["total_order_values"] + model_df["total_freight_value"]
)

model_df[["n_payments","max_installments","total_payment_value"]].isna().sum()


In [ ]:
model_df

In [ ]:
# comparing the two features to check if there is any possibilty of multicollinearity 
# CHECKING THE CORRELATION
model_df["order_total_plus_shipping"] = (
    model_df["total_order_values"] + model_df["total_freight_value"]
)

model_df[["total_payment_value", "order_total_plus_shipping"]].corr()

In [ ]:
# DROP THE UNNECESSAEY order_total_plus_shipping", "total_payment_value"
# BECAUSE "total_order_values", "total_freight_value" PROVIDE GRANUALR INFORMATION FOR A MODLE  
model_df = model_df.drop(columns=["order_total_plus_shipping", "total_payment_value"])

In [ ]:
customers=pd.read_csv("C:/Users/lahar/olist-ml-enterprise/data/raw/olist_customers_dataset.csv")
print(customers.head(10))
print(customers.info())
print(customers.nunique())
print(customers.shape)

In [ ]:
# the two numbers are matching 
'''
These two numbers should match.

Why?

Because if they do not match, then customer_id is not unique in customers, and merging could duplicate your order rows.'''
#Step 1: Check whether every orders.customer_id exists in customers.customer_id
orders["customer_id"].isin(customers["customer_id"]).value_counts()

In [ ]:
print("rows in customers:", len(customers))
print("unique customer_id:", customers["customer_id"].nunique())

In [ ]:
customers.isnull().sum()

##### Step 7 — Now we are ready to create customer features

But before merging, we follow the feature engineering discipline:

We create a separate feature table.

In [ ]:
# here we are creating this seperate tablr to avoid disturabnves with other features
# if we want to use other coulmns like customer_unique_id,customer_zip_code_prefix in future then still we can use it 
'''
customer_unique_id	keep for history later	identifies real customer
customer_zip_code_prefix	maybe later	geographic info
'''
customer_features = customers[[
    "customer_id",
    "customer_state"
]].copy()
print(customer_features.head(10))
print(customer_features.shape)

In [ ]:
#model_df rows stay unchanged so it protects TRAINING DATASET
'''the left join keeps all rows from model_df, and because the join key was clean, it did not duplicate any order rows'''
model_df = model_df.merge(customer_features, on="customer_id", how="left")
#CHECKING IF THE MERGE DOESNOT BREAK THE SPINE TABLE(MODEL DF-TARINING TABLE)
print("rows:", len(model_df))
print("unique order_id:", model_df["order_id"].nunique())

In [ ]:
model_df["customer_state"].isnull().sum()

In [ ]:
customers.groupby("customer_unique_id")["customer_id"].count().value_counts().sort_index()

In [ ]:
customer_order_counts = customers.groupby("customer_unique_id")["customer_id"].count().reset_index()
customer_order_counts.columns = ["customer_unique_id", "customer_total_orders"]

customer_order_counts.head()

In [ ]:
# ADDING customer order counts to customer_features
customers = customers.merge(customer_order_counts, on="customer_unique_id", how="left")
#Recreating customer  feature table (UPDATED)
customer_features = customers[[
    "customer_id",
    "customer_state",
    "customer_total_orders"
]].copy()
# merging it to model df(SPINE TABLE)
model_df = model_df.merge(customer_features, on="customer_id", how="left")

In [ ]:
print(model_df)

⚠️ Important issue (VERY IMPORTANT)

WE merged  customer features again like this directly:

👉 You might get duplicate columns like:

customer_state_x
customer_state_y


In [ ]:
# dropping one column 
model_df = model_df.drop(columns=["customer_state_y"])
model_df = model_df.rename(columns={"customer_state_x": "customer_state"})

In [ ]:
print(model_df)

Quick improvement (important)
Instead of raw values:
1, 2, 3, 5
We often convert to:
model_df["is_repeat_customer"] = (model_df["customer_total_orders"] > 1).astype(int)
Now:
customer_total_orders	is_repeat_customer
1	0
2	1
3	1
👉 This is often more useful for models
Your next step
1.	Fix duplicate columns:
model_df = model_df.drop(columns=["customer_state_y"])
model_df = model_df.rename(columns={"customer_state_x": "customer_state"})
2.	Create new feature:
model_df["is_repeat_customer"] = (model_df["customer_total_orders"] > 1).astype(int)
3.	Show:
model_df[["customer_total_orders", "is_repeat_customer"]].head(10)



In [ ]:
model_df["is_repeat_customer"] = (model_df["customer_total_orders"] > 1).astype(int)

In [ ]:
model_df[["customer_total_orders", "is_repeat_customer"]].head(10)

In [ ]:
#percentage of repeat vs non-repeat customers
model_df["is_repeat_customer"].value_counts(normalize=True)

In [ ]:
products=pd.read_csv("C:/Users/lahar/olist-ml-enterprise/data/raw/olist_products_dataset.csv")
print(products.head())
print(products.info())
print(products.describe())
print(products.isnull().sum())

In [ ]:
# checking if the product id is unique in products 
products["product_id"].nunique() == len(products)

In [ ]:
print("products rows:", len(products))
print("unique product_id in products:", products["product_id"].nunique())
#Check whether product_id is a primary key in products
print(products["product_id"].duplicated().sum())
#Check whether one order has many products 
order_product_counts = order_items.groupby("order_id")["product_id"].nunique().sort_values(ascending=False)
print(order_product_counts.head(10))
print("Orders with more than 1 unique product:", (order_product_counts > 1).sum())

In [ ]:
#Check whether one product appears in many orders
product_order_counts = order_items.groupby("product_id")["order_id"].nunique().sort_values(ascending=False)

print(product_order_counts.head(10))
print("Products appearing in more than 1 order:", (product_order_counts > 1).sum())

In [ ]:
#Check whether same product appears multiple times in the same order
same_product_same_order = (
    order_items
    .groupby(["order_id", "product_id"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print(same_product_same_order.head(10))
print("Order-product pairs repeated more than once:", (same_product_same_order["count"] > 1).sum())

In [ ]:
# dealing with product cleaning first
products_clean = products.copy()
# creating a copy so that we do not modify the original products table directly
products_clean = products.copy()

# ------------------------------------------------------------
# Why are we cleaning this table first?
# ------------------------------------------------------------
# The products table contains product-level information.
# Later, we will merge this table with order_items using product_id.
# After merging, we will aggregate to order level using order_id.
#
# Before doing that, we want to clean missing values in product columns
# so that the aggregation step gives meaningful results.
#
# Our logic from discussion:
# 1. Some missing values carry business meaning
#    Example:
#    - missing photo count can mean no photos were provided
#    - missing description length can mean no description was provided
#    - missing category can mean unknown category
#
# 2. Some missing values are very rare and do not have a safe replacement
#    Example:
#    - product weight and dimensions
#    These are physical properties, and replacing them with 0 would be wrong.
#    Since only 2 rows are missing these values, we can safely drop those rows.
# ------------------------------------------------------------


# ------------------------------------------------------------
# Fill missing values in listing-quality related columns
# ------------------------------------------------------------

# If product_photos_qty is missing, we treat it as 0.
# Reason:
# From a customer/viewpoint, missing photo information behaves like
# no useful photos were provided in the listing.
products_clean["product_photos_qty"] = products_clean["product_photos_qty"].fillna(0)

# If product_description_lenght is missing, we treat it as 0.
# Reason:
# Missing description means no usable product description was provided.
products_clean["product_description_lenght"] = products_clean["product_description_lenght"].fillna(0)

# If product_name_lenght is missing, we also treat it as 0.
# Reason:
# Missing product name length means name information is unavailable.
# Since this column is already a length column, filling with 0 is reasonable.
products_clean["product_name_lenght"] = products_clean["product_name_lenght"].fillna(0)


# ------------------------------------------------------------
# Fill missing category values
# ------------------------------------------------------------

# If product_category_name is missing, we do NOT want to drop the row,
# because category can still be useful as a feature later.
# Also, we do not want to assign a random existing category.
# So we mark it clearly as "unknown".
products_clean["product_category_name"] = products_clean["product_category_name"].fillna("unknown")


# ------------------------------------------------------------
# Drop rows with missing physical attributes
# ------------------------------------------------------------

# product_weight_g, product_length_cm, product_height_cm, and product_width_cm
# describe the physical size/weight of the product.
#
# We should NOT fill these with 0 because:
# - a product cannot realistically have 0 meaningful dimensions
# - filling with fake values would introduce wrong information
#
# Since only 2 rows are missing these columns, dropping them is a simple and safe choice.
products_clean = products_clean.dropna(subset=[
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
])


# ------------------------------------------------------------
# Final check after cleaning
# ------------------------------------------------------------

# check missing values after cleaning
print(products_clean.isnull().sum())

# check shape after cleaning
print("Shape before cleaning:", products.shape)
print("Shape after cleaning:", products_clean.shape)

# preview cleaned table
products_clean.head()

In [ ]:
# ------------------------------------------------------------
# STEP 2: Merge order_items with cleaned products table
# ------------------------------------------------------------

# Why are we doing this merge?
# ------------------------------------------------------------
# order_items contains:
# - order_id
# - product_id
# - item-level purchase information
#
# products_clean contains:
# - product_id
# - product attributes such as photos, description length, weight, dimensions, category
#
# We want to enrich each order_items row with product information.
# This gives us an item-level enriched table.
#
# Important:
# After this merge, the table is STILL NOT at order level.
# It is still at item level, because one order can have multiple item rows.
# We will aggregate to order level in the next step.
# ------------------------------------------------------------

# We merge on product_id because:
# - product_id is the common key between order_items and products_clean
# - product_id is unique in products_clean
# - the same product_id can appear many times in order_items

# We use a LEFT JOIN because:
# - order_items is our base table here
# - we want to keep all purchased items
# - even if product information is missing for some reason, we do not want to lose the order_items row

order_items_products = order_items.merge(
    products_clean,
    on="product_id",
    how="left"
)

# ------------------------------------------------------------
# Sanity checks after merge
# ------------------------------------------------------------

# check shape
print("order_items shape before merge:", order_items.shape)
print("order_items_products shape after merge:", order_items_products.shape)

# check whether number of rows stayed the same
# if yes, then merge did not create unexpected duplication
print("Rows preserved after merge:", len(order_items) == len(order_items_products))

# preview merged table
order_items_products.head()

In [ ]:
# ------------------------------------------------------------
# STEP 3: Aggregate enriched item-level table to order level
# ------------------------------------------------------------

# Why are we aggregating?
# ------------------------------------------------------------
# order_items_products is still at item level.
# That means one order_id can appear multiple times.
#
# But our modeling dataset must have:
# 1 row = 1 order
#
# So now we group by order_id and summarize product-related columns
# into order-level features.
# ------------------------------------------------------------

# Features from product_photos_qty:
# - avg_photos: typical number of photos across products in the order
# - min_photos: worst listed product in the order
# - max_photos: best listed product in the order
#
# Features from product_weight_g:
# - total_weight: total weight of all items in the order
# - max_weight: heaviest single item in the order
# - avg_weight: typical item weight in the order

order_product_features = order_items_products.groupby("order_id").agg(
    avg_photos=("product_photos_qty", "mean"),
    min_photos=("product_photos_qty", "min"),
    max_photos=("product_photos_qty", "max"),

    total_weight=("product_weight_g", "sum"),
    max_weight=("product_weight_g", "max"),
    avg_weight=("product_weight_g", "mean")
).reset_index()

# ------------------------------------------------------------
# Sanity checks after aggregation
# ------------------------------------------------------------

# after aggregation, each order_id should appear only once
print("rows:", len(order_product_features))
print("unique order_id:", order_product_features["order_id"].nunique())

# if these two match, then we successfully created
# a one-row-per-order feature table
print("1 row per order:", len(order_product_features) == order_product_features["order_id"].nunique())

# preview output
order_product_features.head()

In [ ]:
# ------------------------------------------------------------
# STEP 3B: Add category-based order-level feature
# ------------------------------------------------------------

# Why this feature?
# ------------------------------------------------------------
# product_category_name is categorical text.
# We cannot directly keep raw category names in model_df because
# our final table must stay at 1 row = 1 order.
#
# So instead of keeping raw category text, we summarize category
# information at the order level.
#
# n_unique_categories tells us how many different product categories
# are present in one order.
#
# Example:
# ["toys", "books", "toys"] -> unique categories = 2
#
# This can help because:
# - 1 category may indicate a simple order
# - many categories may indicate a more mixed/complex order
# ------------------------------------------------------------

category_features = order_items_products.groupby("order_id").agg(
    n_unique_categories=("product_category_name", "nunique")
).reset_index()

# preview
category_features.head()
# merge category feature into the existing product feature table
order_product_features = order_product_features.merge(
    category_features,
    on="order_id",
    how="left"
)

# quick check
order_product_features.head()

In [ ]:
# ------------------------------------------------------------
# STEP 3C: Create product volume feature
# ------------------------------------------------------------

# Why are we creating volume?
# ------------------------------------------------------------
# product_length_cm, product_height_cm, and product_width_cm
# together describe the physical size of the product.
#
# Instead of treating them as 3 separate raw columns immediately,
# we first combine them into one meaningful size feature:
#
# product_volume_cm3 = length * height * width
#
# This helps us capture overall package size / space occupied.
# That may affect shipping complexity, handling difficulty,
# and possibly customer satisfaction.
# ------------------------------------------------------------

order_items_products["product_volume_cm3"] = (
    order_items_products["product_length_cm"]
    * order_items_products["product_height_cm"]
    * order_items_products["product_width_cm"]
)

# preview a few rows
order_items_products[[
    "order_id",
    "product_id",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume_cm3"
]].head()

In [ ]:
# ------------------------------------------------------------
# STEP 3D: Aggregate product volume to order level
# ------------------------------------------------------------

# Why these features?
# ------------------------------------------------------------
# total_volume -> total space occupied by all items in the order
# max_volume   -> largest single item in the order (worst-case)
# avg_volume   -> typical item size in the order
# ------------------------------------------------------------

volume_features = order_items_products.groupby("order_id").agg(
    total_volume=("product_volume_cm3", "sum"),
    max_volume=("product_volume_cm3", "max"),
    avg_volume=("product_volume_cm3", "mean")
).reset_index()

# preview
volume_features.head()
# merge volume features into order_product_features
order_product_features = order_product_features.merge(
    volume_features,
    on="order_id",
    how="left"
)

# preview final product feature table
order_product_features.head()

In [ ]:
# ------------------------------------------------------------
# STEP 4: Merge full product feature block into model_df
# ------------------------------------------------------------

# Why are we merging now?
# ------------------------------------------------------------
# model_df is our main modeling dataset.
# It already follows:
# 1 row = 1 order
#
# order_product_features also now follows:
# 1 row = 1 order
#
# Since both tables have the same grain, we can safely merge them
# on order_id without creating duplicate rows.
# ------------------------------------------------------------

model_df = model_df.merge(
    order_product_features,
    on="order_id",
    how="left"
)

# ------------------------------------------------------------
# Sanity checks after merge
# ------------------------------------------------------------

# total number of rows
print("Rows after merge:", len(model_df))

# number of unique orders
print("Unique order_id after merge:", model_df["order_id"].nunique())

# check whether still one row = one order
print("Still 1 row per order:", len(model_df) == model_df["order_id"].nunique())

# check missing values in newly added product features
print(
    model_df[
        [
            "avg_photos",
            "min_photos",
            "max_photos",
            "total_weight",
            "max_weight",
            "avg_weight",
            "n_unique_categories",
            "total_volume",
            "max_volume",
            "avg_volume"
        ]
    ].isnull().sum()
)

# preview
model_df.head()

In [ ]:
# ------------------------------------------------------------
# STEP 5: Handle missing product features
# ------------------------------------------------------------

# Why are we dropping rows?
# ------------------------------------------------------------
# Some orders do not have corresponding product features.
# This results in NaN values after merging.
#
# These rows likely:
# - have no valid product information
# - or were affected during cleaning
#
# Instead of filling with fake values (mean/0),
# we drop them because:
# - they are very small in number (16 out of ~95k)
# - they do not provide reliable information for training
# ------------------------------------------------------------

model_df = model_df.dropna(subset=[
    "avg_photos",
    "min_photos",
    "max_photos",
    "avg_weight",
    "max_weight",
    "avg_volume",
    "max_volume"
])

# ------------------------------------------------------------
# Sanity check after dropping
# ------------------------------------------------------------

print("Rows after dropping:", len(model_df))
print("Unique order_id:", model_df["order_id"].nunique())
print("Still 1 row per order:", len(model_df) == model_df["order_id"].nunique())

In [ ]:
model_df

Raw order_purchase_timestamp is not directly useful for modeling because it hides patterns inside a full datetime value. Instead, we extract interpretable time-based features such as hour, weekday, weekend flag, and month.

Next step

Now we should create:

purchase_hour
purchase_dayofweek
is_weekend
purchase_month

In [ ]:
# ------------------------------------------------------------
# STEP: Create time-based features from order_purchase_timestamp
# ------------------------------------------------------------

# convert to datetime if needed
model_df["order_purchase_timestamp"] = pd.to_datetime(model_df["order_purchase_timestamp"])

# extract hour of purchase
model_df["purchase_hour"] = model_df["order_purchase_timestamp"].dt.hour

# extract day of week
# Monday = 0, Sunday = 6
model_df["purchase_dayofweek"] = model_df["order_purchase_timestamp"].dt.dayofweek

# create weekend flag
model_df["is_weekend"] = model_df["purchase_dayofweek"].isin([5, 6]).astype(int)

# extract month of purchase
model_df["purchase_month"] = model_df["order_purchase_timestamp"].dt.month

# preview
model_df[[
    "order_purchase_timestamp",
    "purchase_hour",
    "purchase_dayofweek",
    "is_weekend",
    "purchase_month"
]].head()

In [ ]:
model_df["estimated_delivery_days"] = (
    model_df["order_estimated_delivery_date"] - 
    model_df["order_purchase_timestamp"]
).dt.days

In [ ]:
model_df = model_df.drop(columns=["order_purchase_timestamp"])

In [ ]:
model_df

In [ ]:
model_df["estimated_delivery_days"].describe()
model_df["estimated_delivery_days"].isna().sum()

In [ ]:
#CREATING ANOTHER MODEL_DF SO THAT WE CAN USE IT WITHOUT DISTURBING ACTUAL 
model_df_clean = model_df.copy()

In [ ]:
model_df_clean = model_df_clean.drop(columns=[
    "order_id",
    "customer_id",
    "order_status",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_time_days",
    "review_score",
    "customer_total_orders"
])

In [ ]:
print(model_df_clean.columns)
print(model_df_clean.shape)

In [ ]:
model_df_clean.isna().sum().sort_values(ascending=False)

In [ ]:
model_df_clean

In [ ]:
#Step 5.2.1 — First: look at distributions
model_df_clean.describe()

## Visualisation 

🧠 Why do we even do visualization in ML?

Before modeling, visualization = understanding the data’s behavior.

You are NOT just plotting for fun.

You are asking:

👉 “How does this data behave in reality?”

🎯 Your Goal in Visual EDA

When you plot something, your brain should automatically ask:

1. What is the distribution?
Is it balanced or imbalanced?
Is it normal or skewed?

👉 This affects:

Model performance
Need for transformations (log, scaling)
2. Are there outliers?
Are there extreme values?

👉 This affects:

Model stability
Need for capping / clipping
3. What is the shape of the data?
Symmetric?
Long tail?
Concentrated?

👉 This tells:

Whether mean is reliable
Whether median is better
4. For target variable specifically:

You ask:

👉 “What am I trying to predict, and how hard is it?”

Before plotting, your mindset should be:

👉 “I want to understand how difficult this prediction problem is.”

So you check:

✔️ Is the data balanced?
Example:
90% = 0
10% = 1

👉 This means:

Model may just predict 0 always
Accuracy becomes misleading

In [ ]:
# Visualize the target variable to understand the class imbalance
import matplotlib.pyplot as plt
import pandas as pd
# Count of each class
low_review_counts = model_df_clean['low_review'].value_counts().sort_index()
print(low_review_counts)
# Bar plot here width=6 and height=4
plt.figure(figsize=(6,4))
low_review_counts.plot(kind='bar')

plt.title('Count of low_review')
# for x axis and y axis 
plt.xlabel('low_review')
plt.ylabel('Count')
# to keep the label straight and not tilted 
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Convert to percentage
low_review_pct = model_df_clean['low_review'].value_counts(normalize=True).sort_index()

print(low_review_pct)

# Plot
plt.figure(figsize=(6,4))
low_review_pct.plot(kind='bar')

plt.title('Percentage of low_review')
plt.xlabel('low_review')
plt.ylabel('Percentage')
plt.xticks(rotation=0)
plt.show()

Before plotting any feature, your brain should go:

👉

What type is this?
What plot should I use?
What am I looking for?
What ML problem can this cause?

🟢 Continuous

total_order_values
total_freight_value
avg_weight, max_weight, total_weight
avg_volume, max_volume, total_volume

🔵 Discrete (counts)

n_items
n_unique_products
n_unique_sellers
n_payments
max_installments
n_unique_categories
avg_photos, min_photos, max_photos
estimated_delivery_days

🟡 Binary (categorical)

low_review ✅ (target)
is_repeat_customer
is_weekend

🟠 Ordinal / Cyclical

purchase_hour
purchase_dayofweek
purchase_month

🔴 Categorical

customer_state

#### TRYING A WRONG GRAOH FOR NUMERIC VARIABLE: ESTIMATED DEILVEY DAYS 

#### UNNECESAARY

In [ ]:
plt.figure(figsize=(10,4))

model_df_clean['estimated_delivery_days'].value_counts().plot(kind='bar')

plt.title('Bar plot of estimated_delivery_days')
plt.xlabel('estimated_delivery_days')
plt.ylabel('Count')

plt.show()

In [ ]:
# HERE WE ARE PLOTTING ESTIMATED DEIVERY DAYS

import matplotlib.pyplot as plt  # import plotting library

# Create a figure (controls size of the plot)
plt.figure(figsize=(10,4))

# Plot histogram of delivery_time_days
# bins=50 → divide data into 50 intervals (bars)
plt.hist(model_df_clean['estimated_delivery_days'], bins=500)

# Add title to the plot
plt.title('Distribution of estimated_delivery_days')

# Label x-axis (what variable we are plotting)
plt.xlabel('estimated_delivery_days')

# Label y-axis (how many observations fall in each bin)
plt.ylabel('Frequency')

# Show the plot
plt.show()

In [ ]:
#We confirm this using boxplot
plt.figure(figsize=(10,4))

plt.boxplot(model_df_clean['estimated_delivery_days'])

plt.title('Bar plot of estimated_delivery_days')
plt.ylabel('estimated_delivery_days')

plt.show()

Median = 23, Q1 = 18, Q3 = 28
So median is between Q1 and Q3
Then how can you say “shifted left”?

👉 Good question.

🔥 Key idea (MOST IMPORTANT)

👉 We are NOT checking:

❌ “Is median between Q1 and Q3?” (it ALWAYS is)

👉 We are checking:

✅ Is median closer to Q1 or Q3?

📊 Let’s calculate from your numbers
Q1 = 18
Median = 23
Q3 = 28
Distance check:

👉 Distance from Q1:

23 - 18 = 5

👉 Distance from Q3:

28 - 23 = 5
🤯 What does this mean?

👉 Median is perfectly centered

❗ So correction:

Earlier I said:

“median shifted left”

👉 That is NOT correct based on your numbers

✔ Good catch by you

🧠 Then why does plot look right-skewed?

Because:

👉 Skewness is NOT only about median position

It is also about:

Tail length
Outliers
Spread beyond Q3
🔥 What is happening in YOUR data

From boxplot:

Box (Q1–Q3) → roughly symmetric ✔
BUT
Right side has MANY outliers and long tail ❗
🎯 Correct interpretation

👉
“Median is roughly centered within the box (Q1–Q3), but the distribution is still right-skewed due to a long right tail and many high-value outliers.”

🧠 Important concept (VERY IMPORTANT)
There are TWO types of skew signals:
1️⃣ Inside the box (IQR symmetry)
Median centered → looks symmetric locally
2️⃣ Outside the box (tails & outliers)
Long tail on right → skewed globally

👉 Your data:

Component	Behavior
Inside box	symmetric
Outside (tail)	right-skewed
🔥 Final takeaway (VERY IMPORTANT)

👉
“Median position tells about symmetry inside the central 50%, but skewness is determined by the overall distribution including tails and outliers.”

In [ ]:
import matplotlib.pyplot as plt
# Create a plotting area with a chosen size
plt.figure(figsize=(6,4))
# Draw histogram for total_order_values
# This shows how the values are distributed
# bins=50 means the values are grouped into 50 intervals
plt.hist(model_df_clean['total_order_values'], bins=30)

# Add a title to the graph
plt.title('Distribution of total_order_values')

# Label for x-axis: the feature name
plt.xlabel('total_order_values')

# Label for y-axis: how many rows fall into each interval
plt.ylabel('Frequency')

# Display the graph
plt.show()

In [ ]:
import matplotlib.pyplot as plt  # plotting library

# Create figure
plt.figure(figsize=(10,8))

# Boxplot for total_order_values
plt.boxplot(model_df_clean['total_order_values'])

# Title and label
plt.title('Boxplot of total_order_values')
plt.ylabel('total_order_values')

# Show plot
plt.show()

In [ ]:
# Feature- total_frieght_value
import matplotlib.pyplot as plt  # library for plotting

# Create a figure with size
plt.figure(figsize=(6,4))

# Histogram for total freight value
# bins=50 → divide values into 50 groups
plt.hist(model_df_clean['total_freight_value'], bins=15)

# Add title
plt.title('Distribution of total_freight_value')

# Label x-axis
plt.xlabel('total_freight_value')

# Label y-axis
plt.ylabel('Frequency')

# Show the plot
plt.show()

In [ ]:
import matplotlib.pyplot as plt  # plotting library

# Create figure
plt.figure(figsize=(10,8))

# Boxplot for total_order_values
plt.boxplot(model_df_clean['total_freight_value'])

# Title and label
plt.title('Boxplot of total_freight_values')
plt.ylabel('total_freight_values')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for number of items
# Even though it's discrete, histogram still works
plt.hist(model_df_clean['n_items'], bins=30)

# Title
plt.title('Distribution of n_items')

# X-axis label
plt.xlabel('n_items')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:

plt.figure(figsize=(6,4))
# Plot histogram for n_payments
# bins=10 is enough because values are small integers
plt.hist(model_df_clean['n_payments'], bins=100)

# Add title
plt.title('Distribution of n_payments')

# Label x-axis
plt.xlabel('n_payments')

# Label y-axis
plt.ylabel('Frequency')

# Show the plot
plt.show()

# USE THIS ===> THIS IS RIGHT 
plt.hist(model_df_clean['n_payments'], bins=5)

# Add title
plt.title('Distribution of n_payments')

# Label x-axis
plt.xlabel('n_payments')

# Label y-axis
plt.ylabel('Frequency')

# Show the plot
plt.show()

In [ ]:
# MAX_INSTALLEMENTS
# Create figure
plt.figure(figsize=(6,4))

# Histogram for max_installments
# Using fewer bins because values are small integers
plt.hist(model_df_clean['max_installments'], bins=10)

# Title
plt.title('Distribution of max_installments')

# X-axis label
plt.xlabel('max_installments')

# Y-axis label
plt.ylabel('Frequency')

# Show the plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for avg_photos
# Using moderate bins since values are not too large
plt.hist(model_df_clean['avg_photos'], bins=30)

# Title
plt.title('Distribution of avg_photos')

# X-axis label
plt.xlabel('avg_photos')

# Y-axis label
plt.ylabel('Frequency')

# Show the plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for min_photos
# Using moderate bins
plt.hist(model_df_clean['min_photos'], bins=20)

# Title
plt.title('Distribution of min_photos')

# X-axis label
plt.xlabel('min_photos')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# here mst of the value are between 0 and 2  
# so i am verifying to see if 0 is dominant
model_df_clean['min_photos'].value_counts().sort_index()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for max_photos
# Moderate bins since values can go higher
plt.hist(model_df_clean['max_photos'], bins=30)

# Title
plt.title('Distribution of max_photos')

# X-axis label
plt.xlabel('max_photos')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

model_df_clean['max_photos'].value_counts().sort_index()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for total_weight
# Using more bins because values can vary widely
plt.hist(model_df_clean['total_weight'], bins=30)

# Title
plt.title('Distribution of total_weight')

# X-axis label
plt.xlabel('total_weight')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(10,8))

# Boxplot for total_order_values
plt.boxplot(model_df_clean['total_weight'])

# Title and label
plt.title('Boxplot of total_weight_values')
plt.ylabel('total_weight_values')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for avg_weight
# Using more bins since values can vary
plt.hist(model_df_clean['avg_weight'], bins=40)

# Title
plt.title('Distribution of avg_weight')

# X-axis label
plt.xlabel('avg_weight')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for max_weight
# Using more bins since values vary widely
plt.hist(model_df_clean['max_weight'], bins=40)

# Title
plt.title('Distribution of max_weight')

# X-axis label
plt.xlabel('max_weight')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for purchase_hour
# 24 bins because there are 24 hours in a day
plt.hist(model_df_clean['purchase_hour'], bins=24)

# Title
plt.title('Distribution of purchase_hour')

# X-axis label
plt.xlabel('purchase_hour')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
plt.figure(figsize=(6,4))

# Histogram for purchase_dayofweek
# 7 bins because there are 7 days in a week
plt.hist(model_df_clean['purchase_dayofweek'], bins=7)

# Title
plt.title('Distribution of purchase_dayofweek')

# X-axis label
plt.xlabel('purchase_dayofweek (0=Mon, 6=Sun)')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Count how many 0s and 1s
is_weekend_counts = model_df_clean['is_weekend'].value_counts().sort_index()

# Print counts
print(is_weekend_counts)

# Plot bar chart
plt.figure(figsize=(6,4))
is_weekend_counts.plot(kind='bar')

# Title
plt.title('Count of is_weekend')

# X-axis label
plt.xlabel('is_weekend (0=weekday, 1=weekend)')

# Y-axis label
plt.ylabel('Count')

# Keep labels straight
plt.xticks(rotation=0)

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for purchase_month
# 12 bins because 12 months
plt.hist(model_df_clean['purchase_month'], bins=12)

# Title
plt.title('Distribution of purchase_month')

# X-axis label
plt.xlabel('purchase_month (1=Jan, 12=Dec)')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for estimated delivery days
# Using enough bins to see distribution clearly
plt.hist(model_df_clean['estimated_delivery_days'], bins=40)

# Title
plt.title('Distribution of estimated_delivery_days')

# X-axis label
plt.xlabel('estimated_delivery_days')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for number of unique products
plt.hist(model_df_clean['n_unique_products'], bins=10)

# Title
plt.title('Distribution of n_unique_products')

# X-axis label
plt.xlabel('n_unique_products')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for number of unique sellers
# Small bins because values are small integers
plt.hist(model_df_clean['n_unique_sellers'], bins=10)

# Title
plt.title('Distribution of n_unique_sellers')

# X-axis label
plt.xlabel('n_unique_sellers')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create figure
plt.figure(figsize=(6,4))

# Histogram for number of unique categories
# Using small bins since values are small integers
plt.hist(model_df_clean['n_unique_categories'], bins=10)

# Title
plt.title('Distribution of n_unique_categories')

# X-axis label
plt.xlabel('n_unique_categories')

# Y-axis label
plt.ylabel('Frequency')

# Show plot
plt.show()

In [ ]:
# Create 3 plots side by side
plt.figure(figsize=(18,5))

# Plot 1: total_volume
plt.subplot(1,3,1)
plt.hist(model_df_clean['total_volume'], bins=50)
plt.title('total_volume')

# Plot 2: avg_volume
plt.subplot(1,3,2)
plt.hist(model_df_clean['avg_volume'], bins=50)
plt.title('avg_volume')

# Plot 3: max_volume
plt.subplot(1,3,3)
plt.hist(model_df_clean['max_volume'], bins=50)
plt.title('max_volume')

# Adjust layout
plt.tight_layout()

# Show plots
plt.show()

In [ ]:
# Get counts of each state
state_counts = model_df_clean['customer_state'].value_counts()

# Take top 10 states for readability
top_states = state_counts.head(10)

# Plot
plt.figure(figsize=(8,4))
top_states.plot(kind='bar')

# Title
plt.title('Top 10 customer states')

# X-axis label
plt.xlabel('customer_state')

# Y-axis label
plt.ylabel('Count')

# Rotate labels for readability
plt.xticks(rotation=45)

In [ ]:
# Count values
repeat_counts = model_df_clean['is_repeat_customer'].value_counts().sort_index()

# Print counts
print(repeat_counts)

# Plot bar chart
plt.figure(figsize=(6,4))
repeat_counts.plot(kind='bar')

# Title
plt.title('Count of is_repeat_customer')

# X-axis label
plt.xlabel('is_repeat_customer (0=No, 1=Yes)')

# Y-axis label
plt.ylabel('Count')

# Keep labels straight
plt.xticks(rotation=0)

# Show plot
plt.show()

In [ ]:
import sys
print(sys.executable)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Select only numeric columns
numeric_cols = model_df_clean.select_dtypes(include=np.number)

# Correlation matrix
corr_matrix = numeric_cols.corr()

# Plot heatmap
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap of Numeric Features")
plt.show()


In [ ]:
plt.figure(figsize=(14, 10))

sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0,
    annot=False,          # keep False for now (numbers will clutter)
    linewidths=0.5,       # 🔥 THIS adds lines between cells
    linecolor='black'     # 🔥 black grid lines
)

plt.title("Correlation Heatmap (Clear View)")
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.show()

In [ ]:
# Correlation with target
target_corr = corr_matrix['low_review'].sort_values(ascending=False)

print(target_corr)

Step-by-step:
1️⃣ Sign (direction)

Positive (+) → as feature increases → bad reviews increase

Negative (−) → as feature increases → bad reviews decrease
2️⃣ Strength (importance)

Value	Meaning

0.00 – 0.05	no signal

0.05 – 0.15	weak

0.15 – 0.30	moderate

> 0.30	strong
> 
3️⃣ Final sentence (THIS is what you say)

👉 “Higher number of items is associated with slightly higher chance of bad reviews.”

DONE.

🚨 VERY IMPORTANT RULE

👉 Correlation ≠ causation
👉 Correlation = tendency / association

So always say:

❌ “causes bad reviews”

✅ “is associated with higher chance of bad reviews”

In [ ]:
import sys
!{sys.executable} -m pip install statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
# Select numeric features
X = model_df_clean.select_dtypes(include='number').drop(columns=['low_review'])

# Create VIF dataframe
vif_df = pd.DataFrame()
vif_df['feature'] = X.columns
vif_df['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

# Sort
vif_df = vif_df.sort_values(by='VIF', ascending=False)

print(vif_df)

In [ ]:
cols_to_drop = [
    'min_photos', 'max_photos',
    'max_weight',
    'max_volume',
    'is_weekend'
]

model_df_reduced = model_df_clean.drop(columns=cols_to_drop)

print("Remaining columns:")
print(model_df_reduced.columns)

In [ ]:
model_df_reduced.isna().sum().sort_values(ascending=False)

In [ ]:
model_df_reduced[['total_weight', 'avg_weight', 'total_volume', 'avg_volume']].describe()

In [ ]:
print("Number of unique states:", model_df_reduced['customer_state'].nunique())
print("\nUnique states:")
print(sorted(model_df_reduced['customer_state'].unique()))

In [ ]:
model_df_encoded=pd.get_dummies(model_df_reduced,columns=['customer_state'],drop_first=True)
print(model_df_encoded.shape)

In [ ]:
model_df_reduced.shape

In [ ]:
model_df_encoded